# Multidimensional GBM Monte Carlo Template

This notebook is ordered so you can plug in:

1. your model inputs
2. your covariance matrix
3. the function you want to evaluate on each simulated path

The core idea is:

- simulate correlated Brownian shocks using the covariance matrix
- evolve all assets jointly under a multidimensional GBM
- apply your payoff / objective function to each simulated path


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Define simulation inputs

Keep all user-editable assumptions in one place.


In [ ]:
# Basic Monte Carlo settings
n_assets = 3
n_sims = 10_000
n_steps = 252
T = 1.0
dt = T / n_steps

# Optional: name each dimension for readability
asset_names = ["HF_Price", "iBAP_Price", "throughput"]

# Initial values S(0)
S0 = np.array([100.0, 80.0, 120.0])

# Annualized drifts mu
mu = np.array([0.08, 0.05, 0.06])

# Annualized covariance matrix of the GBM drivers
# Replace this with your own covariance matrix
cov_matrix = np.array([
    [0.000804, 0.001032, 0],
    [0.001032, 0.017536, 0],
    [0, 0, 0]
], dtype=float)

assert len(S0) == n_assets
assert len(mu) == n_assets
assert cov_matrix.shape == (n_assets, n_assets)

## 2. Validate covariance matrix

This helps catch dimension errors and non-positive semidefinite inputs early.


In [ ]:
def validate_covariance_matrix(cov):
    cov = np.asarray(cov, dtype=float)

    if cov.ndim != 2 or cov.shape[0] != cov.shape[1]:
        raise ValueError("cov_matrix must be square.")

    if not np.allclose(cov, cov.T, atol=1e-10):
        raise ValueError("cov_matrix must be symmetric.")

    eigenvalues = np.linalg.eigvalsh(cov)
    if np.min(eigenvalues) < -1e-10:
        raise ValueError("cov_matrix must be positive semidefinite.")

    return eigenvalues


eigenvalues = validate_covariance_matrix(cov_matrix)
vols = np.sqrt(np.diag(cov_matrix))
corr_matrix = cov_matrix / np.outer(vols, vols)

print("Eigenvalues:", eigenvalues)
print("Volatilities:", vols)
print("Correlation matrix:\n", corr_matrix)

## 3. Multidimensional GBM simulator

The update rule is:

$$S_{t+\Delta t}^{(i)} = S_t^{(i)} \exp\left((\mu_i - 0.5\sigma_i^2)\Delta t + \Delta W_i\right)$$

where the correlated increment vector satisfies:

$$\Delta W \sim \mathcal{N}(0, \Sigma \Delta t)$$

and `cov_matrix = Sigma`.


In [ ]:
def simulate_multidim_gbm(S0, mu, cov_matrix, T, n_steps, n_sims, random_seed=42):
    S0 = np.asarray(S0, dtype=float)
    mu = np.asarray(mu, dtype=float)
    cov_matrix = np.asarray(cov_matrix, dtype=float)

    n_assets = len(S0)
    dt = T / n_steps

    validate_covariance_matrix(cov_matrix)

    rng = np.random.default_rng(random_seed)

    # Use an eigenvalue factorization to remain stable even if the matrix
    # is only positive semidefinite.
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    eigenvalues = np.clip(eigenvalues, 0.0, None)
    cov_sqrt = eigenvectors @ np.diag(np.sqrt(eigenvalues)) @ eigenvectors.T

    z = rng.normal(size=(n_sims, n_steps, n_assets))
    dW = np.sqrt(dt) * np.einsum("tka,ab->tkb", z, cov_sqrt)

    variances = np.diag(cov_matrix)
    drift_term = (mu - 0.5 * variances) * dt

    log_returns = drift_term[None, None, :] + dW

    log_paths = np.zeros((n_sims, n_steps + 1, n_assets), dtype=float)
    log_paths[:, 0, :] = np.log(S0)
    log_paths[:, 1:, :] = np.log(S0)[None, None, :] + np.cumsum(log_returns, axis=1)

    paths = np.exp(log_paths)
    return paths

## 4. Define the function you want to simulate

Replace this with your actual function. The input `paths` has shape:

`(n_sims, n_steps + 1, n_assets)`


In [ ]:
def evaluate_simulated_function(paths):
    # Example: basket value at maturity
    terminal_values = paths[:, -1, :]
    weights = np.array([0.4, 0.3, 0.3])
    basket_terminal = terminal_values @ weights

    # Replace the line below with your own function output
    return basket_terminal

## 5. Run the simulation


In [ ]:
paths = simulate_multidim_gbm(
    S0=S0,
    mu=mu,
    cov_matrix=cov_matrix,
    T=T,
    n_steps=n_steps,
    n_sims=n_sims,
    random_seed=42,
)

results = evaluate_simulated_function(paths)

print("paths shape:", paths.shape)
print("results shape:", results.shape)

## 6. Summarize results


In [ ]:
summary = pd.Series(results).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
summary

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(results, bins=50, edgecolor="black", alpha=0.7)
plt.title("Distribution of simulated function values")
plt.xlabel("Function value")
plt.ylabel("Frequency")
plt.show()

## 7. Inspect a few simulated paths


In [ ]:
time_grid = np.linspace(0, T, n_steps + 1)

fig, axes = plt.subplots(n_assets, 1, figsize=(9, 3 * n_assets), sharex=True)
if n_assets == 1:
    axes = [axes]

for asset_idx, ax in enumerate(axes):
    for sim_idx in range(min(20, n_sims)):
        ax.plot(time_grid, paths[sim_idx, :, asset_idx], alpha=0.6)
    ax.set_title(f"Simulated paths for {asset_names[asset_idx]}")
    ax.set_ylabel("Value")

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

## Notes

- If your covariance matrix is estimated from historical log returns, it can be used directly as `cov_matrix`.
- If you instead have a correlation matrix and separate volatilities, build covariance as:
  `cov_matrix = np.outer(vols, vols) * corr_matrix`
- If your function depends on the full path rather than only terminal values, use the full `paths` array inside `evaluate_simulated_function`.
- If you send me your exact function and covariance matrix, I can wire them into this notebook directly.
